# Preprocess walkthrough

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Preprocess Walkthrough

This notebook demonstrates discovery of site files, generation of a composite bathymetry, building static features (distance, bearing, bathy diff), and constructing a 10m wind field interpolated to NORA3 points. Run cells sequentially.

In [ ]:
from pathlib import Path
import sys, os

os.chdir("..")

# Now import the preprocess helper
from src import preprocess

mapping = preprocess.discover_site_files("configs/sites.yaml")
print("Discovered nearshore sites:", len(mapping))
if mapping:
    name = next(iter(mapping))
    info = mapping[name]
    print("Example site:", name)
    print("Target files found:", len(info["target_files"]))
    print("Paired offshore files found:", len(info["paired_offshore_files"]))
    print("Target sample path:", info["target_files"][0] if info["target_files"] else "None")

## Generate / load composite bathymetry (optional if already created)

The generator script `src/preprocess/generate_bathy_field.py` will create a composite `.npz` from the `.xyz` tiles. If you already ran it earlier and have `experiments/bathy_composite_test.npz`, you can skip this cell.

In [ ]:
import subprocess, sys, pathlib

out = pathlib.Path("data/processed/bathy_composite_trondheimsfjord.npz")
if out.exists():
    print(out, "already exists; skip generation")
else:
    print("Generating composite bathy...")
    subprocess.check_call(
        [sys.executable, "src/preprocess/generate_bathy_field.py", "--out", str(out)]
    )
    print("Saved", out)

## Build static features (distance, bearing, bathy diff)

This runs `src/preprocess/static_build.py` and writes `experiments/static_features.csv`.

In [ ]:
from pathlib import Path
import subprocess, sys, csv
from IPython.display import display

wide_path = Path("data/processed/static_features_wide.csv")
out = wide_path if wide_path.exists() else Path("data/processed/static_features.csv")
print("Static features path:", out)
if not out.exists():
    print("Generating static features...")
    subprocess.check_call(
        [
            sys.executable,
            "src/preprocess/static_build.py",
            "--bathy",
            "data/processed/bathy_composite_trondheimsfjord.npz",
            "--out",
            str(out),
        ]
    )
    # prefer wide after generation
    if wide_path.exists():
        out = wide_path

# Try pandas first, fallback to pure-CSV pretty-print
use_pandas = False
try:
    import pandas as pd

    use_pandas = True
except Exception as e:
    print("pandas not available; using fallback CSV reader:", e)

if use_pandas:
    df = pd.read_csv(out)
    print("Data shape:", df.shape)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    display(df.head(30))
    print("...")
else:
    # fallback: read CSV and pretty-print in plain text
    with open(out, newline="") as fh:
        reader = csv.DictReader(fh)
        rows = list(reader)
        columns = reader.fieldnames or []
    print("Rows:", len(rows))
    if len(rows) == 0:
        print("No rows found")
    else:
        # determine column widths (cap to keep lines reasonable)
        max_col_width = 40
        widths = []
        for c in columns:
            maxw = max(len(str(c)), max((len(str(r.get(c, ""))) for r in rows), default=0))
            widths.append(min(maxw, max_col_width))

        def format_row(row):
            cells = []
            for i, c in enumerate(columns):
                v = str(row.get(c, ""))
                if len(v) > widths[i]:
                    v = v[: widths[i] - 3] + "..."
                cells.append(v.ljust(widths[i]))
            return " | ".join(cells)

        header = " | ".join(c.ljust(widths[i]) for i, c in enumerate(columns))
        sep = "-+-".join("-" * widths[i] for i in range(len(columns)))
        print(header)
        print(sep)
        for r in rows[:30]:
            print(format_row(r))
        # grouped view per NORA3 (for wide, each row is a NORA3)
        if "nora3_name" in columns:
            from collections import defaultdict

            groups = defaultdict(list)
            for r in rows:
                groups[r["nora3_name"]].append(r)
            for name in groups:
                print(f"\n=== NORA3: {name} ({len(groups[name])} pairs) ===")
                print(header)
                print(sep)
                for r in groups[name][:10]:
                    print(format_row(r))

## Split, Normalize, and Inspect Preprocessed Datasets

This cell assembles NORAC timeseries, performs a chronological per-site split, normalizes numeric features (fit on train only), and writes normalized CSVs to `experiments/preprocessed/`.

In [ ]:
print("Preview: raw NORAC (nearshore) and NORA3 (offshore) parameter samples before normalization")
from src import preprocess
from pathlib import Path
from IPython.display import display

try:
    import pandas as pd

    _has_pd = True
except Exception as e:
    print("pandas not available:", e)
    _has_pd = False

mapping = preprocess.discover_site_files("configs/sites.yaml")
print("Discovered sites:", len(mapping))
if not mapping:
    print("No sites found; aborting preview")
else:
    # pick first site that has at least one NORAC or NORA3 file
    site_name, info = next(
        (
            (k, v)
            for k, v in mapping.items()
            if (v.get("target_files") or v.get("paired_offshore_files"))
        ),
        (None, None),
    )
    if site_name is None:
        print("No site with data found")
    else:
        print("Example site:", site_name)
        # Preview NORAC timeseries assembled from target_files
        try:
            df = preprocess._assemble_timeseries({site_name: info}) if _has_pd else None
        except Exception as e:
            print("Error assembling timeseries for", site_name, e)
            df = None
        if df is None or (hasattr(df, "empty") and df.empty):
            print("No NORAC timeseries rows found for", site_name)
        else:
            print("\nNORAC timeseries sample (first 8 rows):")
            if _has_pd:
                pd.set_option("display.max_columns", None)
                pd.set_option("display.width", 200)
                display(df.head(8))
            else:
                print(df.head(8))

        # Preview NORA3 (offshore) parameter file(s)
        paired = info.get("paired_offshore_files") or []
        if not paired:
            print("\nNo paired NORA3 files for this site")
        else:
            print("\nPaired NORA3 files:", len(paired))
            for fp in paired[:2]:
                print("\nNORA3 sample from", fp)
                try:
                    if _has_pd:
                        pddf = pd.read_csv(fp, comment="#")
                        pd.set_option("display.max_columns", None)
                        pd.set_option("display.width", 200)
                        display(pddf.head(8))
                    else:
                        with open(fp) as fh:
                            for _ in range(8):
                                line = fh.readline()
                                if not line:
                                    break
                                print(line.rstrip())
                except Exception as e:
                    print("Failed to read", fp, e)

In [ ]:
print("Building datasets: split and normalize")
from src import preprocess

res = preprocess.build_and_normalize_datasets(
    sites_yaml="configs/sites.yaml", training_config="configs/training.yaml"
)
print("Results:", res)

In [ ]:
from pathlib import Path
import pandas as pd

base = Path("experiments/preprocessed")
for split in ["train", "val", "test"]:
    p = base / f"{split}_norm.csv"
    print("\n", p)
    if p.exists():
        df = pd.read_csv(p)
        print(split, "shape=", df.shape)
        display(df.head(10))
    else:
        print("not found")

In [ ]:
from pathlib import Path

try:
    import torch
except Exception as e:
    print("torch not available:", e)
else:
    base = Path("experiments/preprocessed")
    for split in ["train", "val", "test"]:
        p = base / f"{split}_norm.csv"
        if not p.exists():
            continue
        df = pd.read_csv(p)
        num_cols = df.select_dtypes(include=["number"]).columns.tolist()
        arr = df[num_cols].values.astype(float)
        t = torch.tensor(arr)
        torch.save(t, base / f"{split}_norm.pt")
        print("Saved tensor:", base / f"{split}_norm.pt")

## Build wind field (10m) and preview interpolated values at NORA3 wave points

This runs `src/preprocess/wind_field.py` and writes `experiments/wind_field_nora3_10m.npz`.

In [ ]:
from pathlib import Path
import subprocess, sys, numpy as np

out = Path("experiments/wind_field_nora3_10m.npz")
print("Running wind field builder...")
subprocess.check_call([sys.executable, "src/preprocess/wind_field.py", "--out", str(out)])
print("Wrote", out)
data = np.load(str(out), allow_pickle=True)
print("site count:", data["site_names"].shape[0])
print("sample:")
for i in range(min(8, data["site_names"].shape[0])):
    print(
        data["site_names"][i],
        "speed=",
        round(float(data["wind_speed"][i]), 2),
        "m/s dir=",
        round(float(data["wind_dir"][i]), 1),
    )